# BMW AI — Module 1D: Train YOLOv8n Driver Monitoring (Colab)

Fine-tune **YOLOv8n** on the Kaggle DMS dataset:
[habbas11/dms-driver-monitoring-system](https://www.kaggle.com/datasets/habbas11/dms-driver-monitoring-system)

**Classes (5):**
| id | name |
|----|------|
| 0 | Open Eye |
| 1 | Closed Eye |
| 2 | Cigarette |
| 3 | Phone |
| 4 | Seatbelt |

**Runtime:** Runtime → Change runtime type → **GPU (T4)**

After training, download `best.pt` and save it locally as:
`F:/BMW/ml/models/driver_monitor_best.pt`

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade ultralytics opendatasets pyyaml

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Mount Google Drive (optional but recommended)

Saves checkpoints if the Colab session disconnects.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/bmw_ai")
MODEL_DIR = DRIVE_ROOT / "models"
RUNS_DIR = DRIVE_ROOT / "runs"

for p in (MODEL_DIR, RUNS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("MODEL_DIR:", MODEL_DIR)
print("RUNS_DIR :", RUNS_DIR)

## 3. Download Kaggle DMS dataset

Uses [opendatasets](https://pypi.org/project/opendatasets/).  
You need a Kaggle account + `kaggle.json` API token when prompted:
Kaggle → Account → Create New API Token.

In [ ]:
import opendatasets as od
from pathlib import Path

dataset_url = "https://www.kaggle.com/datasets/habbas11/dms-driver-monitoring-system"
od.download(dataset_url)

DATASET_ROOT = Path("/content/dms-driver-monitoring-system")
assert DATASET_ROOT.is_dir(), f"Dataset folder not found: {DATASET_ROOT}"
print("DATASET_ROOT:", DATASET_ROOT)
print("Contents:", sorted(p.name for p in DATASET_ROOT.iterdir()))

## 4. Write `data.yaml`

In [ ]:
from pathlib import Path
import yaml

DATASET_ROOT = Path("/content/dms-driver-monitoring-system")
YAML_PATH = DATASET_ROOT / "data.yaml"

# Roboflow-style layout used by this Kaggle dataset
train_rel = "train/images" if (DATASET_ROOT / "train" / "images").exists() else "images/train"
val_rel = "valid/images" if (DATASET_ROOT / "valid" / "images").exists() else "images/val"

yaml_data = {
    "path": str(DATASET_ROOT.resolve()),
    "train": train_rel,
    "val": val_rel,
    "nc": 5,
    "names": [
        "Open Eye",
        "Closed Eye",
        "Cigarette",
        "Phone",
        "Seatbelt",
    ],
}

with open(YAML_PATH, "w", encoding="utf-8") as outfile:
    yaml.dump(yaml_data, outfile, default_flow_style=False, sort_keys=False)

print(f"--- {YAML_PATH.name} ---")
print(YAML_PATH.read_text())

n_train = len(list((DATASET_ROOT / train_rel).glob("*.*")))
n_val = len(list((DATASET_ROOT / val_rel).glob("*.*")))
print(f"train images: {n_train} | val images: {n_val}")
assert n_train > 0, "No training images found"

## 5. Train YOLOv8n

~2–5 hours on Colab T4 for 100 epochs. Set `EPOCHS = 5` for a smoke test.

If you hit CUDA OOM, lower `BATCH` to `16` or `8`.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

EPOCHS = 100
BATCH = 32
IMGSZ = 640
NAME = "driver_monitor_dms_v1"

try:
    PROJECT = str(RUNS_DIR)
except NameError:
    PROJECT = "/content/runs"

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=15,
    save_period=10,
    project=PROJECT,
    name=NAME,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

print("\nTraining finished!")
if hasattr(results, "results_dict"):
    metrics = results.results_dict
    print(f"Best mAP50    : {metrics.get('metrics/mAP50(B)', 'N/A')}")
    print(f"Best mAP50-95 : {metrics.get('metrics/mAP50-95(B)', 'N/A')}")
print("Model saved to:", results.save_dir)

## 6. Copy `best.pt` to Drive + download locally

On your PC, save the file as:
`F:/BMW/ml/models/driver_monitor_best.pt`

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

best = Path(results.save_dir) / "weights" / "best.pt"
assert best.is_file(), f"best.pt not found at {best}"

try:
    dest = MODEL_DIR / "driver_monitor_best.pt"
    shutil.copy2(best, dest)
    print("Copied to Drive:", dest)
except NameError:
    dest = Path("/content/driver_monitor_best.pt")
    shutil.copy2(best, dest)
    print("Copied to:", dest)

files.download(str(dest))
print("Download started — place file at ml/models/driver_monitor_best.pt in the BMW repo")

## 7. Quick inference smoke test (optional)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

weights = Path(dest)
model = YOLO(str(weights))
print("Class names:", model.names)

val_dir = DATASET_ROOT / "valid" / "images"
train_dir = DATASET_ROOT / "train" / "images"
sample_images = list(val_dir.glob("*.*")) if val_dir.is_dir() else []
if not sample_images and train_dir.is_dir():
    sample_images = list(train_dir.glob("*.*"))

if sample_images:
    pred = model.predict(str(sample_images[0]), conf=0.25, verbose=False)
    print("Sample:", sample_images[0].name)
    print("Detections:", len(pred[0].boxes) if pred[0].boxes is not None else 0)
else:
    print("No sample images found")